# GeoGuess: Country Prediction from Street-View Images

**Authors:** Jimin Lee, Juheon Kim, Sara Petrosian, Tyson Johnson, Heeseung Moon

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tjohns94/geolocation-project/blob/main/geolocation_guesser.ipynb)

---

## Project Overview

This notebook contains the complete, reproducible pipeline for our group project:

1. **Model Training** (Part A) — Fine-tune EfficientNet-B0 on the OpenStreetView-5M dataset to predict country from street-view images
2. **Data Collection** (Part B) — How we built a Flask webapp to collect human guesses from our case study group
3. **Analysis** (Part C) — Statistical comparison of model vs. human accuracy using McNemar's test

### Research Question

> *To what extent can an EfficientNet-B0 model outperform a case study group of participants at identifying the country of origin of street-view photographs?*

### Hypotheses

- **H₀**: There is no statistically significant difference between the accuracy of the model and the case study participants.
- **Hₐ**: The model will achieve significantly higher accuracy than the case study participants.

### Run Modes

This notebook supports three modes, controlled by the `MODE` variable in the first code cell:

| Mode | What it does | Time | GPU needed? |
|------|-------------|------|-------------|
| `"analysis_only"` | Loads pre-computed results and runs all analysis (Part C) | ~1 min | No |
| `"evaluate"` | Downloads OSV-5M shards, loads checkpoint, runs test-set evaluation + analysis | ~30 min | Yes |
| `"train"` | Full training from scratch + evaluation + analysis | ~2 hrs | Yes (A100 recommended) |

**Default is `"analysis_only"`** — just hit Runtime → Run all.

### Key Results

| Entity | Accuracy | N |
|--------|----------|---|
| EfficientNet-B0 | **64.70%** | 1,000 |
| Old model | 32.10% | 1,000 |
| Group (any correct) | 20.60% | 1,000 |
| Group (majority vote) | 12.70% | 1,000 |
| Best individual (Jimin) | 15.28% | 360 |
| Random baseline | 0.62% | — |

All McNemar tests reject H₀ at α = 0.05 (p < 10⁻⁴⁰ for individuals, p < 10⁻⁷⁰ for group).

## Setup

The cell below automatically clones the GitHub repository if the data files
are not already present. It works on both Google Colab and local environments.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# MODE SELECTION — change this to control what the notebook runs
# ════════════════════════════════════════════════════════════════════════════
MODE = "analysis_only"   # "analysis_only" | "evaluate" | "train"
# ════════════════════════════════════════════════════════════════════════════

import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/tjohns94/geolocation-project.git"
REPO_NAME = "geolocation-project"

# Detect environment
IN_COLAB = "google.colab" in sys.modules

# Auto-clone repo if data files are missing
if not Path("data/experiment_data.json").exists():
    if Path(REPO_NAME).exists():
        os.chdir(REPO_NAME)
        print(f"Changed directory to {REPO_NAME}/")
    else:
        print(f"Cloning {REPO_URL} ...")
        subprocess.run(["git", "clone", REPO_URL], check=True)
        os.chdir(REPO_NAME)
        print(f"Cloned and changed directory to {REPO_NAME}/")

# Verify required files exist
required = ["data/experiment_data.json"]
if MODE != "analysis_only":
    required.append("data/best_model_efficientnet_b0.pt")
for f in required:
    assert Path(f).exists(), f"Missing required file: {f}"

print(f"Mode: {MODE}")
print(f"Working directory: {os.getcwd()}")
print(f"Required files: all present")

---
# Part A: Model Training Pipeline

This section contains the complete training code for our EfficientNet-B0 country classifier,
fine-tuned on ~500,000 images from [OpenStreetView-5M](https://huggingface.co/datasets/osv5m/osv5m).

> **In `"analysis_only"` mode, this entire section is skipped.** The pre-computed model
> predictions are loaded directly in Part C.

### Why ~500,000 Training Images?

The full OpenStreetView-5M dataset contains over 5 million images, but we limited training
to ~500K (10 shards) for two reasons:

1. **Colab resource limits** — Google Colab sessions have a maximum runtime of ~12 hours
   and limited disk space. Downloading and processing the full dataset would exceed these
   constraints.
2. **Diminishing returns** — Our training curves show validation loss plateauing by epoch 5–6
   (see below), suggesting additional data would yield marginal accuracy improvements for
   significantly more compute cost.

The model converged to **75.0% validation top-1 accuracy** and **57.6% test accuracy** on
the full held-out evaluation set. On the 1,000 webapp test images used for Part C, the model
achieves 64.7% accuracy.

In [ ]:
# Display saved training artifacts (produced during original A100 training run)
from IPython.display import display, Image as IPImage
from pathlib import Path

artifacts = {
    "Training curves (loss and accuracy per epoch)": "outputs/training_curves.png",
    "Confusion matrix (top classes)": "outputs/confusion_matrix.png",
    "Sample predictions on test images": "outputs/sample_predictions.png",
}

for title, path in artifacts.items():
    if Path(path).exists():
        print(f"\n{title}:")
        display(IPImage(filename=path, width=700))
    else:
        print(f"\n{title}: [file not found at {path}]")

# Training history
history_path = Path("outputs/training_history.csv")
if history_path.exists():
    history = pd.read_csv(history_path)
    print("\nTraining history (6 epochs on NVIDIA A100):")
    display(history.style.format({
        "train_loss": "{:.3f}", "val_loss": "{:.3f}",
        "train_acc": "{:.1%}", "val_top1_acc": "{:.1%}",
    }).set_caption("EfficientNet-B0 training on ~500K OSV-5M images"))

### Country Classification: 180 Classes

The training pipeline builds its label vocabulary from the training set, resulting in
**180 unique country classes** (ISO 3166-1 alpha-2 codes). This is more than the 161
countries present in the 1,000 webapp test images — some countries appear only in the
training data. The label mapping is saved alongside the checkpoint to ensure consistency.

## §A1 — Environment Setup

In [ ]:
if MODE != "analysis_only":
    # Install dependencies
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "datasets<4.0.0", "huggingface_hub", "timm", "torchmetrics",
                    "accelerate", "scikit-learn", "pandas", "numpy", "matplotlib",
                    "pillow", "tqdm", "h3", "pycountry", "pyyaml", "seaborn", "scipy"],
                   check=True)
    print("All packages installed.")
else:
    print("Skipping Part A (MODE = analysis_only). Jump to Part C.")

## §A2 — Configuration

All hyperparameters are defined here. The notebook reads from `configs/training.yaml`
if available; otherwise it uses inline defaults.

In [ ]:
if MODE != "analysis_only":
    import yaml
    from pathlib import Path

    DEFAULTS = {
        "task_mode": "country",
        "data": {
            "n_shards": 10, "train_samples": 500_000, "val_samples": 15_000,
            "test_samples": 5_000, "min_images_per_class": 20, "max_countries": None,
        },
        "model": {"architecture": "efficientnet_b0", "image_size": 224, "pretrained": True},
        "training": {
            "batch_size": 32, "epochs": 6, "learning_rate": 3e-4,
            "weight_decay": 1e-4, "patience": 3, "num_workers": 2, "seed": 42,
        },
        "geocell": {"h3_resolution": 3},
        "io": {
            "output_dir": "/content/osv5m_outputs",
            "use_drive": True,
            "drive_output_dir": "/content/drive/MyDrive/osv5m_outputs",
        },
        "webapp_prep": {"n_images": 1000, "seed": 42, "jpeg_quality": 85, "output_dir": "webapp/data"},
        "webapp_holdout_ids": "configs/webapp_holdout_ids.json",
    }

    CONFIG_PATH = "configs/training.yaml"
    _cfg = DEFAULTS.copy()
    for candidate in [CONFIG_PATH, "/content/drive/MyDrive/osv5m_outputs/training.yaml"]:
        p = Path(candidate)
        if p.exists():
            with open(p) as f:
                _yaml_cfg = yaml.safe_load(f)
            if _yaml_cfg:
                for key, val in _yaml_cfg.items():
                    if isinstance(val, dict) and isinstance(_cfg.get(key), dict):
                        _cfg[key] = {**_cfg[key], **val}
                    else:
                        _cfg[key] = val
                print(f"Config loaded from: {p}")
                break
    else:
        print("No YAML config found -- using inline defaults.")

    TASK_MODE   = _cfg["task_mode"]
    N_SHARDS    = _cfg["data"]["n_shards"]
    TRAIN_SAMPLES = _cfg["data"]["train_samples"]
    VAL_SAMPLES = _cfg["data"]["val_samples"]
    TEST_SAMPLES = _cfg["data"]["test_samples"]
    MIN_IMAGES_PER_CLASS = _cfg["data"]["min_images_per_class"]
    MAX_COUNTRIES = _cfg["data"].get("max_countries")
    MODEL_NAME  = _cfg["model"]["architecture"]
    IMAGE_SIZE  = _cfg["model"]["image_size"]
    USE_PRETRAINED = _cfg["model"].get("pretrained", True)
    BATCH_SIZE  = _cfg["training"]["batch_size"]
    EPOCHS      = _cfg["training"]["epochs"]
    LR          = _cfg["training"]["learning_rate"]
    WEIGHT_DECAY = _cfg["training"]["weight_decay"]
    PATIENCE    = _cfg["training"]["patience"]
    NUM_WORKERS = _cfg["training"]["num_workers"]
    SEED        = _cfg["training"]["seed"]
    H3_RESOLUTION = _cfg["geocell"]["h3_resolution"]
    OUTPUT_DIR  = _cfg["io"]["output_dir"]
    USE_DRIVE   = _cfg["io"]["use_drive"]
    DRIVE_OUTPUT_DIR = _cfg["io"].get("drive_output_dir")
    WEBAPP_HOLDOUT_PATH = _cfg.get("webapp_holdout_ids", "configs/webapp_holdout_ids.json")

    print(f"Task mode   : {TASK_MODE}")
    print(f"Model       : {MODEL_NAME}  (image size: {IMAGE_SIZE})")
    print(f"Data        : {N_SHARDS} shards -> {TRAIN_SAMPLES:,} train / {VAL_SAMPLES:,} val / {TEST_SAMPLES:,} test")
    print(f"Training    : {EPOCHS} epochs, lr={LR}, bs={BATCH_SIZE}, patience={PATIENCE}")

## §A3 — Imports, Seeds & Device

In [ ]:
from __future__ import annotations

if MODE != "analysis_only":
    import json, math, os, random, time, warnings
    from io import BytesIO
    from pathlib import Path
    from typing import Any

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from PIL import Image
    from tqdm.auto import tqdm

    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.amp import autocast, GradScaler
    from torch.utils.data import Dataset, DataLoader
    from torchvision import transforms

    import timm
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score
    from huggingface_hub import hf_hub_download

    try:
        import h3
        H3_AVAILABLE = True
    except ImportError:
        H3_AVAILABLE = False

    try:
        import pycountry
        PYCOUNTRY_AVAILABLE = True
    except ImportError:
        PYCOUNTRY_AVAILABLE = False

    warnings.filterwarnings("ignore")

    def set_seed(seed=42):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

    set_seed(SEED)

    DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    USE_AMP = DEVICE.type == "cuda"

    if DEVICE.type == "cuda":
        print(f"GPU  : {torch.cuda.get_device_name(0)}")
        print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("WARNING: No GPU detected. Set Runtime -> GPU for training/evaluation.")

    def haversine(lat1, lon1, lat2, lon2):
        R = 6371.0
        phi1, phi2 = np.radians(lat1), np.radians(lat2)
        dphi = np.radians(lat2 - lat1)
        dlam = np.radians(lon2 - lon1)
        a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam / 2) ** 2
        return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

    OUTPUT_DIR = str(Path(OUTPUT_DIR) / TASK_MODE)
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    print(f"Outputs -> {OUTPUT_DIR}")

## §A4 — Google Drive (Optional)

Mounting Drive provides shard caching and persistent artifact storage.

In [ ]:
if MODE != "analysis_only":
    DRIVE_OUTPUT_DIR_ACTUAL = None
    DRIVE_CACHE_DIR = None

    if USE_DRIVE:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            _base = _cfg["io"].get("drive_output_dir", "/content/drive/MyDrive/osv5m_outputs")
            DRIVE_OUTPUT_DIR_ACTUAL = str(Path(_base) / TASK_MODE)
            DRIVE_CACHE_DIR = str(Path(_base) / "shard_cache")
            Path(DRIVE_OUTPUT_DIR_ACTUAL).mkdir(parents=True, exist_ok=True)
            Path(DRIVE_CACHE_DIR).mkdir(parents=True, exist_ok=True)
            print(f"Drive mounted. Artifacts -> {DRIVE_OUTPUT_DIR_ACTUAL}")
        except ImportError:
            print("Not running on Colab. Outputs will be local only.")
            USE_DRIVE = False
        except Exception as e:
            print(f"Drive mount failed: {e}. Continuing without Drive.")
            USE_DRIVE = False
    else:
        print("Drive not mounted (use_drive: false).")

## §A5 — Dataset Download

Downloads OSV-5M shards from HuggingFace Hub. Cached to Google Drive if mounted.

In [ ]:
if MODE != "analysis_only":
    # Uncomment and add your HuggingFace token to download OSV-5M shards:
    # os.environ["HF_TOKEN"] = "hf_your_token_here"
    if "HF_TOKEN" not in os.environ:
        raise EnvironmentError(
            "HF_TOKEN not set. Get a token at https://huggingface.co/settings/tokens "
            "and set it via: export HF_TOKEN=hf_your_token_here")

    CACHE_DIR = Path(OUTPUT_DIR) / "osv5m_cache"
    IMG_DIR   = CACHE_DIR / "images"
    _REPO     = "osv5m/osv5m"

    def _hub_dl(filename):
        return Path(hf_hub_download(repo_id=_REPO, repo_type="dataset",
                                     filename=filename, local_dir=str(CACHE_DIR)))

    def _try_copy_from_drive(shard_name, dest_dir):
        if not DRIVE_CACHE_DIR:
            return False
        cached = Path(DRIVE_CACHE_DIR) / shard_name
        if cached.exists():
            import shutil
            shutil.copy2(cached, dest_dir / shard_name.split("/")[-1])
            return True
        return False

    def _cache_to_drive(local_path, shard_name):
        if not DRIVE_CACHE_DIR:
            return
        dest = Path(DRIVE_CACHE_DIR) / shard_name.replace("/", "_")
        if not dest.exists():
            import shutil
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(local_path, dest)

    def download_osv5m(n_train_shards):
        import zipfile
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        for name in ("train.csv", "test.csv"):
            dst = CACHE_DIR / name
            if dst.exists():
                print(f"  {name}: already cached.")
            else:
                print(f"  Downloading {name} ...", end=" ", flush=True)
                _hub_dl(name)
                print("done.")

        shard_plan = [("train", n_train_shards), ("test", 5)]
        for split, n in shard_plan:
            out_dir = IMG_DIR / split
            out_dir.mkdir(parents=True, exist_ok=True)
            for i in range(n):
                shard_name = f"images/{split}/shard-{i:04d}.zip"
                local_zip  = CACHE_DIR / shard_name
                marker     = out_dir / f".shard-{i:04d}-extracted"
                if marker.exists():
                    print(f"  {shard_name}: already extracted.")
                    continue
                if not local_zip.exists():
                    drive_name = shard_name.replace("/", "_")
                    if _try_copy_from_drive(drive_name, local_zip.parent):
                        print(f"  {shard_name}: copied from Drive cache.")
                    else:
                        print(f"  Downloading {shard_name} ...", end=" ", flush=True)
                        local_zip.parent.mkdir(parents=True, exist_ok=True)
                        _hub_dl(shard_name)
                        print("done.")
                        _cache_to_drive(local_zip, drive_name)
                print(f"  Extracting {shard_name} ...", end=" ", flush=True)
                with zipfile.ZipFile(local_zip) as zf:
                    zf.extractall(out_dir)
                marker.touch()
                print("done.")
        print(f"All shards ready. Images in {IMG_DIR}")

    download_osv5m(N_SHARDS)

## §A6 — Build Metadata Manifest

In [ ]:
if MODE != "analysis_only":
    def build_manifest():
        parts = []
        for split in ("train", "test"):
            csv_path = CACHE_DIR / f"{split}.csv"
            if not csv_path.exists():
                print(f"  WARNING: {csv_path} not found.")
                continue
            print(f"  Reading {split}.csv ...", end=" ", flush=True)
            t0 = time.time()
            df_meta = pd.read_csv(csv_path, index_col=0)
            print(f"{len(df_meta):,} rows ({time.time() - t0:.1f}s)")

            cols = {c.lower(): c for c in df_meta.columns}
            lat_col = next((cols[k] for k in ("latitude", "lat") if k in cols), None)
            lon_col = next((cols[k] for k in ("longitude", "lon", "lng") if k in cols), None)
            country_col = next((cols[k] for k in ("country", "iso_code", "iso") if k in cols), None)

            img_dir = IMG_DIR / split
            image_files = sorted(img_dir.rglob("*.jpg")) or sorted(img_dir.rglob("*.jpeg"))
            print(f"  {split}: {len(image_files):,} image files found.")
            id_to_path = {f.stem: str(f) for f in image_files}

            valid_ids = [idx for idx in df_meta.index if str(idx) in id_to_path]
            df_meta = df_meta.loc[valid_ids]
            n = len(df_meta)
            df = pd.DataFrame({
                "split": split,
                "file_path": [id_to_path[str(idx)] for idx in df_meta.index],
                "latitude": df_meta[lat_col].values.astype(np.float32) if lat_col else np.zeros(n, np.float32),
                "longitude": df_meta[lon_col].values.astype(np.float32) if lon_col else np.zeros(n, np.float32),
                "country": df_meta[country_col].values if country_col else [None] * n,
            })
            n_before = len(df)
            df = df.dropna(subset=["latitude", "longitude", "country"])
            df = df[(df["latitude"].between(-90, 90)) & (df["longitude"].between(-180, 180))]
            df = df[df["country"].str.len() == 2]
            print(f"  {split}: {n_before} -> {len(df)} after cleaning")
            parts.append(df)

        manifest = pd.concat(parts, ignore_index=True)
        print(f"Full manifest: {len(manifest):,} rows ({manifest['country'].nunique()} countries)")
        return manifest

    manifest = build_manifest()

## §A7 — Stratified Sampling

Stratified sampling ensures proportional country representation.
Validation is carved from the train split only — no test data leakage.

In [ ]:
if MODE != "analysis_only":
    def stratified_sample(pool, n, stratify_col, seed):
        n = min(n, len(pool))
        if stratify_col and stratify_col in pool.columns and pool[stratify_col].nunique() > 1:
            try:
                sampled = pool.groupby(stratify_col, group_keys=False).apply(
                    lambda g: g.sample(min(len(g), max(1, round(n * len(g) / len(pool)))), random_state=seed))
                return sampled.sample(min(n, len(sampled)), random_state=seed)
            except (ValueError, KeyError):
                pass
        return pool.sample(n, random_state=seed)

    def sample_manifest(manifest, train_n, val_n, test_n, stratify_col=None,
                        max_classes=None, min_per_class=20, seed=42):
        train_pool = manifest[manifest["split"] == "train"].copy()
        test_pool  = manifest[manifest["split"] == "test"].copy()

        if stratify_col and min_per_class > 0:
            counts = train_pool[stratify_col].value_counts()
            keep = counts[counts >= min_per_class].index
            dropped = len(counts) - len(keep)
            train_pool = train_pool[train_pool[stratify_col].isin(keep)]
            test_pool  = test_pool[test_pool[stratify_col].isin(keep)]
            if dropped > 0:
                print(f"  Dropped {dropped} classes with < {min_per_class} train samples.")

        if max_classes:
            top = train_pool[stratify_col].value_counts().head(max_classes).index
            train_pool = train_pool[train_pool[stratify_col].isin(top)]
            test_pool  = test_pool[test_pool[stratify_col].isin(top)]

        combined = stratified_sample(train_pool, train_n + val_n, stratify_col, seed)
        df_val   = combined.sample(min(val_n, len(combined)), random_state=seed + 1)
        df_train = combined.drop(df_val.index)
        df_test  = stratified_sample(test_pool, test_n, stratify_col, seed + 2)

        assert len(set(df_train.index) & set(df_val.index)) == 0, "Train/val overlap!"
        assert len(set(df_train.index) & set(df_test.index)) == 0, "Train/test overlap!"

        for name, df in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
            nc = df[stratify_col].nunique() if stratify_col else "N/A"
            print(f"  {name:5s}: {len(df):>7,} samples, {nc} classes")
        return df_train.reset_index(drop=True), df_val.reset_index(drop=True), df_test.reset_index(drop=True)

    df_train, df_val, df_test = sample_manifest(
        manifest, TRAIN_SAMPLES, VAL_SAMPLES, TEST_SAMPLES,
        stratify_col="country", max_classes=MAX_COUNTRIES,
        min_per_class=MIN_IMAGES_PER_CLASS, seed=SEED)

## §A8 — Label Preparation

Country strings are encoded to integer IDs. Vocabulary is built from the **training set only**.

In [ ]:
if MODE != "analysis_only":
    def prepare_labels(df_train, df_val, df_test):
        train_countries = sorted(df_train["country"].dropna().unique())
        country2id = {c: i for i, c in enumerate(train_countries)}
        id2country = list(train_countries)

        def map_labels(df):
            df = df.copy()
            df["label"] = df["country"].map(country2id).fillna(-1).astype(int)
            return df[df["label"] >= 0].reset_index(drop=True)

        df_train = map_labels(df_train)
        df_val   = map_labels(df_val)
        df_test  = map_labels(df_test)
        num_classes = len(train_countries)
        label_meta = {"id2country": id2country, "country2id": country2id}
        print(f"Country mode: {num_classes} classes")
        top = df_train["country"].value_counts().head(10)
        for c, n in top.items():
            print(f"  {c}: {n:,}")
        return df_train, df_val, df_test, label_meta, num_classes

    df_train, df_val, df_test, LABEL_META, NUM_CLASSES = prepare_labels(df_train, df_val, df_test)

## §A9 — PyTorch Dataset and DataLoaders

Train uses random crops, horizontal flips, and color jitter. Eval uses deterministic center crops.
Both use ImageNet normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]).

In [ ]:
if MODE != "analysis_only":
    _MEAN = [0.485, 0.456, 0.406]
    _STD  = [0.229, 0.224, 0.225]

    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
        transforms.ToTensor(),
        transforms.Normalize(_MEAN, _STD),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize(int(IMAGE_SIZE * 1.143)),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(_MEAN, _STD),
    ])

    class GeoDataset(Dataset):
        def __init__(self, manifest, transform):
            self.manifest = manifest.reset_index(drop=True)
            self.transform = transform
            self._n_skipped = 0
        def __len__(self):
            return len(self.manifest)
        def __getitem__(self, i):
            row = self.manifest.iloc[i]
            label = int(row["label"])
            meta = {"latitude": float(row.get("latitude", 0.0)),
                    "longitude": float(row.get("longitude", 0.0)),
                    "country": str(row.get("country", ""))}
            try:
                img = Image.open(str(row["file_path"])).convert("RGB")
                return self.transform(img), label, meta
            except (OSError, IOError, ValueError):
                self._n_skipped += 1
                return None

    def collate_skip_nones(batch):
        batch = [b for b in batch if b is not None]
        if not batch:
            return None
        return torch.utils.data.dataloader.default_collate(batch)

    _nw = min(NUM_WORKERS, max(1, len(os.sched_getaffinity(0)))) if hasattr(os, "sched_getaffinity") else NUM_WORKERS
    ds_train = GeoDataset(df_train, train_tf)
    ds_val   = GeoDataset(df_val,   eval_tf)
    ds_test  = GeoDataset(df_test,  eval_tf)

    loader_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=_nw, collate_fn=collate_skip_nones,
                              pin_memory=True, persistent_workers=_nw > 0)
    loader_val   = DataLoader(ds_val, batch_size=BATCH_SIZE * 2, shuffle=False,
                              num_workers=_nw, collate_fn=collate_skip_nones,
                              pin_memory=True, persistent_workers=_nw > 0)
    loader_test  = DataLoader(ds_test, batch_size=BATCH_SIZE * 2, shuffle=False,
                              num_workers=_nw, collate_fn=collate_skip_nones,
                              pin_memory=True, persistent_workers=_nw > 0)
    print(f"Datasets: train={len(ds_train):,}  val={len(ds_val):,}  test={len(ds_test):,}")

## §A10 — Model: EfficientNet-B0

EfficientNet-B0 (~5.3M parameters) uses compound scaling of depth, width, and resolution,
achieving better accuracy-per-FLOP than ResNet or DenseNet alternatives.

In [ ]:
if MODE != "analysis_only":
    def build_model(num_classes, model_name="efficientnet_b0"):
        model = timm.create_model(model_name, pretrained=USE_PRETRAINED, num_classes=num_classes)
        total = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Model: {model_name}  |  {total:,} params  |  {num_classes} classes")
        return model.to(DEVICE)

    model = build_model(NUM_CLASSES, MODEL_NAME)

## §A11 — Training Utilities

AdamW + cosine LR + mixed precision (AMP) + gradient clipping.

In [ ]:
if MODE != "analysis_only":
    def train_one_epoch(model, loader, optimizer, scheduler, scaler, epoch):
        model.train()
        total_loss = correct = total = 0
        pbar = tqdm(loader, desc=f"Epoch {epoch+1} [train]", leave=False)
        for batch in pbar:
            if batch is None: continue
            imgs, labels, _ = batch
            imgs = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits = model(imgs)
                loss = F.cross_entropy(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            bs = labels.size(0)
            total_loss += loss.item() * bs
            correct += (logits.argmax(1) == labels).sum().item()
            total += bs
            pbar.set_postfix(loss=f"{loss.item():.3f}", acc=f"{correct/total:.3f}")
        if scheduler is not None:
            scheduler.step()
        return {"loss": total_loss / max(total, 1), "acc": correct / max(total, 1)}

    @torch.no_grad()
    def evaluate(model, loader, num_classes, split_name="val"):
        model.eval()
        all_labels, all_preds, all_probs = [], [], []
        total_loss = total = 0
        pbar = tqdm(loader, desc=f"Eval [{split_name}]", leave=False)
        for batch in pbar:
            if batch is None: continue
            imgs, labels, meta = batch
            imgs = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            with autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits = model(imgs)
                loss = F.cross_entropy(logits, labels)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            preds = logits.argmax(1).cpu().numpy()
            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.tolist())
            all_probs.extend(probs.tolist())
            total_loss += loss.item() * labels.size(0)
            total += labels.size(0)

        labels_arr = np.array(all_labels)
        preds_arr = np.array(all_preds)
        probs_arr = np.array(all_probs)
        top1 = float((preds_arr == labels_arr).mean())
        top5_correct = sum(1 for i, p in enumerate(probs_arr) if labels_arr[i] in np.argsort(p)[-5:])
        top5_acc = top5_correct / len(labels_arr)

        metrics = {"loss": total_loss / max(total, 1), "top1_acc": top1,
                   "top5_acc": top5_acc, "labels": labels_arr,
                   "preds": preds_arr, "probs": probs_arr}
        if TASK_MODE == "country":
            metrics["macro_f1"] = f1_score(labels_arr, preds_arr, average="macro", zero_division=0)
        return metrics

## §A12 — Checkpoint Loading / Training

- In `"evaluate"` mode: loads the saved checkpoint and skips training.
- In `"train"` mode: trains from scratch with AdamW + cosine annealing + early stopping.

In [ ]:
if MODE != "analysis_only":
    CKPT_FILENAME = f"best_model_{MODEL_NAME}.pt"
    best_ckpt_path = Path(OUTPUT_DIR) / CKPT_FILENAME

    # Search for checkpoint in several locations
    for _candidate in [
        Path(DRIVE_OUTPUT_DIR_ACTUAL or "") / CKPT_FILENAME,
        Path("data") / CKPT_FILENAME,
        Path("data") / "best_model.pt",
        Path("best_model.pt"),
    ]:
        if _candidate.exists() and not best_ckpt_path.exists():
            import shutil
            best_ckpt_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(_candidate, best_ckpt_path)
            print(f"Checkpoint copied from: {_candidate}")
            break

    if MODE == "evaluate":
        assert best_ckpt_path.exists(), f"No checkpoint at {best_ckpt_path}"
        ckpt = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
        assert ckpt["num_classes"] == NUM_CLASSES, (
            f"Checkpoint num_classes={ckpt['num_classes']} != split num_classes={NUM_CLASSES}. "
            "Seed or config mismatch -- data leakage risk.")
        model.load_state_dict(ckpt["model_state"])
        best_val_acc = ckpt["val_top1"]
        best_epoch = ckpt["epoch"]
        _nan = float("nan")
        df_history = pd.DataFrame([{
            "epoch": best_epoch, "train_loss": _nan, "train_acc": _nan,
            "val_loss": _nan, "val_top1_acc": best_val_acc, "val_top5_acc": _nan, "elapsed_s": _nan,
        }])
        print(f"Checkpoint loaded: epoch {best_epoch}, val_top1={best_val_acc:.4f}")
        print("Training skipped. Proceeding to evaluation.")

    elif MODE == "train":
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR / 100)
        scaler = GradScaler(device=DEVICE.type, enabled=USE_AMP)

        best_val_acc = 0.0
        best_epoch = -1
        no_improve = 0
        history = []

        print(f"Training {MODEL_NAME} | {NUM_CLASSES} classes | {EPOCHS} epochs")
        print(f"  optimizer: AdamW lr={LR} wd={WEIGHT_DECAY}")
        print(f"  scheduler: CosineAnnealing T_max={EPOCHS}")
        print(f"  AMP: {USE_AMP} | patience: {PATIENCE}")

        for epoch in range(EPOCHS):
            t0 = time.time()
            ds_train._n_skipped = 0
            train_m = train_one_epoch(model, loader_train, optimizer, scheduler, scaler, epoch)
            val_m = evaluate(model, loader_val, NUM_CLASSES, split_name="val")
            elapsed = time.time() - t0

            row = {"epoch": epoch + 1, "train_loss": train_m["loss"], "train_acc": train_m["acc"],
                   "val_loss": val_m["loss"], "val_top1_acc": val_m["top1_acc"],
                   "val_top5_acc": val_m["top5_acc"], "elapsed_s": elapsed}
            if TASK_MODE == "country":
                row["val_macro_f1"] = val_m.get("macro_f1", 0.0)
            history.append(row)

            improved = val_m["top1_acc"] > best_val_acc
            if improved:
                best_val_acc = val_m["top1_acc"]
                best_epoch = epoch + 1
                no_improve = 0
                ckpt_data = {"model_state": model.state_dict(), "model_name": MODEL_NAME,
                             "num_classes": NUM_CLASSES, "task_mode": TASK_MODE,
                             "epoch": best_epoch, "val_top1": best_val_acc, "config": _cfg}
                if TASK_MODE == "country":
                    ckpt_data["label_meta"] = LABEL_META
                torch.save(ckpt_data, best_ckpt_path)
            else:
                no_improve += 1

            star = " *" if improved else ""
            print(f"Epoch {epoch+1}/{EPOCHS}  val_top1={row['val_top1_acc']:.4f}{star}  ({elapsed:.0f}s)")
            if no_improve >= PATIENCE:
                print(f"Early stopping at epoch {epoch+1}")
                break

        df_history = pd.DataFrame(history)
        print(f"Best: epoch {best_epoch}, val_top1={best_val_acc:.4f}")

## §A13 — Final Evaluation on Test Set

In [ ]:
if MODE != "analysis_only":
    ckpt = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    print(f"Loaded checkpoint: epoch {ckpt['epoch']}  val_top1={ckpt['val_top1']:.4f}")

    test_m = evaluate(model, loader_test, NUM_CLASSES, split_name="test")

    print()
    print("=" * 50)
    print("TEST SET RESULTS")
    print("=" * 50)
    print(f"  Top-1 accuracy : {test_m['top1_acc']:.4f}")
    print(f"  Top-5 accuracy : {test_m['top5_acc']:.4f}")
    if TASK_MODE == "country":
        print(f"  Macro F1       : {test_m.get('macro_f1', float('nan')):.4f}")

## §A14 — Results & Visualization

Training curves, confusion matrix, and sample predictions.

In [ ]:
if MODE != "analysis_only":
    # Training curves (only if we trained from scratch)
    if MODE == "train" and len(df_history) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        ep = df_history["epoch"].values
        axes[0].plot(ep, df_history["train_loss"], marker="o", label="train")
        axes[0].plot(ep, df_history["val_loss"], marker="s", label="val")
        axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
        axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)
        axes[1].plot(ep, df_history["train_acc"], marker="o", label="train top-1")
        axes[1].plot(ep, df_history["val_top1_acc"], marker="s", label="val top-1")
        axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
        axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].grid(alpha=0.3)
        axes[1].set_ylim(0, 1)
        plt.suptitle(f"{MODEL_NAME} | {NUM_CLASSES} classes", fontsize=11)
        plt.tight_layout()
        plt.show()

    # Confusion matrix
    if TASK_MODE == "country":
        id2country = LABEL_META["id2country"]
        labels_arr = test_m["labels"]
        preds_arr = test_m["preds"]

        top_n = min(15, NUM_CLASSES)
        top_ids = pd.Series(labels_arr).value_counts().head(top_n).index.tolist()
        mask = np.isin(labels_arr, top_ids)
        if mask.sum() > 0:
            cm = confusion_matrix(labels_arr[mask], preds_arr[mask], labels=top_ids)
            fig, ax = plt.subplots(figsize=(12, 10))
            ConfusionMatrixDisplay(cm, display_labels=[id2country[i] for i in top_ids]).plot(
                ax=ax, xticks_rotation=45, colorbar=False, values_format="d")
            ax.set_title(f"Confusion matrix - top {top_n} countries (test set)")
            plt.tight_layout()
            plt.show()

    # Sample predictions
    model.eval()
    N_SAMPLES = 8
    sample_rows = df_test.sample(min(N_SAMPLES, len(df_test)), random_state=SEED).reset_index(drop=True)
    id2country = LABEL_META.get("id2country", [])
    fig, axes = plt.subplots(2, N_SAMPLES // 2, figsize=(16, 8))
    axes = axes.flatten()
    for ax_i, row in sample_rows.iterrows():
        try:
            img = Image.open(str(row["file_path"])).convert("RGB")
        except Exception:
            img = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), color=(180, 180, 180))
        tensor = eval_tf(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad(), autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(tensor)
        probs = F.softmax(logits, dim=1).cpu().numpy()[0]
        pred_id = int(np.argmax(probs))
        true_id = int(row["label"])
        conf = float(probs[pred_id])
        title = f"Pred: {id2country[pred_id]}\nTrue: {id2country[true_id]}\nConf: {conf:.1%}"
        color = "green" if pred_id == true_id else "red"
        axes[ax_i].imshow(img)
        axes[ax_i].set_title(title, fontsize=9, color=color)
        axes[ax_i].axis("off")
    plt.suptitle("Sample predictions from test set", fontsize=12)
    plt.tight_layout()
    plt.show()

---
# Part B: Human Data Collection

This section documents how we collected human guesses to compare against the model.
The webapp code is not executable in Colab, so we describe the architecture and
methodology here. The full source code is included in the `webapp/` directory.

## Overview

We built a **Flask webapp** deployed at `https://hanguk.dev/geoguess/` that presents
the same 1,000 test images to human participants. Each participant guesses the country
for as many images as they can, and results are stored in a SQLite database.

## Architecture

```
Browser (Leaflet.js map + country autocomplete)
    |
    v
Flask API (app.py)
    |-- /api/next-image    -> prioritize images other users have seen
    |-- /api/submit-guess  -> validate + store in SQLite
    |-- /api/export        -> download all data as JSON (protected)
    |
    v
SQLite (guesses.db)
    |-- guesses table: username, image_id, guessed_country, is_correct, timestamp
    |-- model_guesses table: image_id, predicted_country, confidence
```

## Key Design Decisions

1. **Image selection priority**: The webapp prioritizes showing each user images that
   other participants have already guessed, maximizing overlap for paired statistical tests.

2. **Country autocomplete**: Users type a country name and select from a dropdown of
   all 161 countries in the dataset (ISO 3166-1 names). This prevents typos and ensures
   fair comparison with the model.

3. **Data isolation**: The 1,000 webapp images come from the OSV-5M **test** split,
   which is entirely separate from training data at the HuggingFace level.

4. **Rate limiting**: 60 requests/min and 20 guesses/min per IP to prevent abuse.

5. **Security**: Input sanitization, CSP headers, X-Frame-Options, parameterized SQL queries.

## Deployment

- **Host**: Ubuntu 22.04 VPS (RackNerd) at `hanguk.dev`
- **Web server**: Apache2 with reverse proxy to Gunicorn (port 5000)
- **SSL**: Cloudflare origin certificate (valid until 2039)
- **Process manager**: systemd service (`geoguess`)

## Case Study Participants

Five group members each independently guessed countries for as many images as possible:

| Participant | Images Guessed |
|-------------|---------------|
| Jimin       | 1,000         |
| Juheon      | 1,000         |
| Tyson       | 285           |
| Sara        | 150           |
| Heeseung    | 144           |

**Total**: 2,579 human guesses across 1,000 images.

## Data Export

Results were exported via the protected API endpoint and merged into
`experiment_data.json`, which feeds directly into Part C below.

---
# Part C: Statistical Analysis — Human vs. Model

This section performs a comprehensive comparison between human case study
participants and our EfficientNet-B0 model using McNemar's test for paired
binary outcomes.

**Data file** (included in the `data/` folder):
- `experiment_data.json` — merged dataset containing 1,000 image records with
  predictions from both model checkpoints (old and EfficientNet-B0) and all
  human guesses from the five case study participants.

## §C1 — Analysis Setup

In [ ]:
import json
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Install scipy if not available (needed for chi-squared p-values)
try:
    from scipy.stats import chi2 as _chi2_dist
    def chi2_sf(stat: float) -> float:
        return float(1 - _chi2_dist.cdf(stat, df=1))
except ImportError:
    try:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scipy"], check=True)
        from scipy.stats import chi2 as _chi2_dist
        def chi2_sf(stat: float) -> float:
            return float(1 - _chi2_dist.cdf(stat, df=1))
    except Exception:
        from math import erfc, sqrt
        def chi2_sf(stat: float) -> float:
            return erfc(sqrt(stat / 2))

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 140)

DATA_PATH = "data/experiment_data.json"

print("Analysis setup complete.")

## §C2 — Load Data

In [ ]:
with open(DATA_PATH) as f:
    experiment = json.load(f)

print(f"Loaded experiment data: {experiment['metadata']['n_images']:,} images, "
      f"{experiment['metadata']['n_human_guesses']:,} human guesses")
print(f"Participants: {', '.join(experiment['metadata']['participants'])}")

# Build images DataFrame from the flat list
images = pd.DataFrame(experiment["images"])
images.rename(columns={
    "old_model_prediction": "old_prediction",
    "old_model_confidence": "old_confidence",
    "old_model_correct": "old_correct",
    "efficientnet_b0_prediction": "new_prediction",
    "efficientnet_b0_confidence": "new_confidence",
    "efficientnet_b0_correct": "new_correct",
}, inplace=True)

# Build guesses DataFrame
guesses = pd.DataFrame(experiment["human_guesses"])
guesses["timestamp"] = pd.to_datetime(guesses["timestamp"])

# Integrity checks
assert images["image_id"].is_unique, "Duplicate image IDs"
assert guesses["image_id"].isin(images["image_id"]).all(), "Orphaned guesses"
assert guesses[["username", "image_id"]].duplicated().sum() == 0, "Duplicate user-image pairs"

print(f"\nHuman guesses: {len(guesses):,} rows")
print(f"Images:        {len(images):,}")
print(f"Users:         {sorted(guesses['username'].unique())}")
print("Integrity checks passed.")

## §C3 — Descriptive Statistics

Per-user summary of guessing activity and accuracy.

In [ ]:
user_summary = (
    guesses.groupby("username")
    .agg(guesses=("is_correct", "size"),
         correct=("is_correct", "sum"))
    .assign(accuracy=lambda df: df["correct"] / df["guesses"])
    .sort_values("guesses", ascending=False)
)
user_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

user_summary["guesses"].plot.bar(ax=axes[0], color="#4c78a8")
axes[0].set_title("Number of guesses per user")
axes[0].set_ylabel("guesses")
axes[0].tick_params(axis="x", rotation=0)

user_summary["accuracy"].plot.bar(ax=axes[1], color="#e45756")
axes[1].set_title("Accuracy per user")
axes[1].set_ylabel("accuracy")
axes[1].set_ylim(0, max(user_summary["accuracy"]) * 1.3)
axes[1].tick_params(axis="x", rotation=0)
for i, (idx, row) in enumerate(user_summary.iterrows()):
    axes[1].text(i, row["accuracy"] + 0.005, f"{row['accuracy']:.1%}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## §C4 — Model Comparison: Old vs. EfficientNet-B0

Our model was improved during the project. The final EfficientNet-B0 was trained on
10 shards (~500K images) compared to the initial checkpoint trained on just 5 shards
(~30K images). McNemar's test confirms the improvement is statistically significant.

In [ ]:
def mcnemar(b: int, c: int, continuity: bool = False) -> dict:
    """McNemar's test on discordant pairs."""
    if b + c == 0:
        return {"chi2": 0.0, "p": 1.0, "b": b, "c": c}
    numer = (abs(b - c) - 1) ** 2 if continuity else (b - c) ** 2
    stat = numer / (b + c)
    return {"chi2": float(stat), "p": chi2_sf(stat), "b": b, "c": c}

old_acc = images["old_correct"].mean()
new_acc = images["new_correct"].mean()

b = int(((images["new_correct"] == 1) & (images["old_correct"] == 0)).sum())
c = int(((images["new_correct"] == 0) & (images["old_correct"] == 1)).sum())
print(f"Old model accuracy:         {old_acc:.1%}")
print(f"EfficientNet-B0 accuracy:   {new_acc:.1%}")
print(f"Absolute improvement:       +{(new_acc - old_acc) * 100:.1f} pp")
print(f"McNemar (old vs EfficientNet-B0): {mcnemar(b, c)}")

In [ ]:
bars = pd.Series({
    "Old model": old_acc,
    "EfficientNet-B0": new_acc,
    **{u: user_summary.loc[u, "accuracy"] for u in user_summary.index},
})

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#aec7e8", "#1f77b4"] + ["#f58518"] * len(user_summary)
bars.plot.bar(ax=ax, color=colors)
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy comparison: models vs. individual humans")
ax.set_ylim(0, max(bars) * 1.15)
for i, v in enumerate(bars):
    ax.text(i, v + 0.005, f"{v:.1%}", ha="center", fontsize=9)
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## §C5 — Individual McNemar Tests: Each Participant vs. Model

McNemar's test compares paired binary outcomes on the same images.
For each participant, we build a 2×2 contingency table of
(human correct, model correct) on the images that participant guessed.

In [ ]:
def paired_table(user_df):
    return user_df.merge(
        images[["image_id", "new_correct"]], on="image_id"
    ).rename(columns={"is_correct": "human_correct"})

rows = []
for user, ug in guesses.groupby("username"):
    pt = paired_table(ug)
    both = int(((pt["human_correct"] == 1) & (pt["new_correct"] == 1)).sum())
    model_only = int(((pt["human_correct"] == 0) & (pt["new_correct"] == 1)).sum())
    human_only = int(((pt["human_correct"] == 1) & (pt["new_correct"] == 0)).sum())
    neither = int(((pt["human_correct"] == 0) & (pt["new_correct"] == 0)).sum())
    test = mcnemar(model_only, human_only)
    rows.append({
        "user": user, "n": len(pt),
        "human_acc": pt["human_correct"].mean(),
        "model_acc": pt["new_correct"].mean(),
        "both_correct": both, "model_only": model_only,
        "human_only": human_only, "both_wrong": neither,
        "chi2": test["chi2"], "p_value": test["p"],
    })

mcnemar_user = pd.DataFrame(rows).sort_values("n", ascending=False).reset_index(drop=True)
mcnemar_user

In [ ]:
plot_df = mcnemar_user.set_index("user")[["both_correct", "model_only", "human_only", "both_wrong"]]
plot_pct = plot_df.div(plot_df.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(10, 5))
plot_pct.plot.barh(stacked=True, ax=ax,
                   color=["#2ca02c", "#1f77b4", "#f58518", "#d62728"])
ax.set_xlabel("Proportion of images")
ax.set_title("Paired outcomes: each participant vs. EfficientNet-B0")
ax.legend(loc="lower right", fontsize=9)

for i, (_, r) in enumerate(mcnemar_user.iterrows()):
    p = r["p_value"]
    star = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    ax.text(1.02, i, f"p={p:.1e} {star}", va="center", fontsize=8, transform=ax.get_yaxis_transform())

plt.tight_layout()
plt.show()

## §C6 — Group-Level Analysis: Three Aggregation Methods

Since each image was guessed by a different subset of participants, we use three
methods to aggregate individual guesses into a single group outcome per image:

1. **Any correct** — group gets credit if *any* participant got it right (oracle)
2. **Majority vote** — group gets credit if >50% of guessers got it right
3. **Average score** — fractional score = proportion of guessers who got it right

In [ ]:
img_group = (
    guesses.groupby("image_id")
    .agg(
        n_guessers=("username", "nunique"),
        n_correct=("is_correct", "sum"),
        n_guesses=("is_correct", "size"),
    )
)
img_group["any_correct"] = (img_group["n_correct"] > 0).astype(int)
img_group["majority_correct"] = (img_group["n_correct"] > img_group["n_guessers"] / 2).astype(int)
img_group["avg_score"] = img_group["n_correct"] / img_group["n_guessers"]

ga = images[["image_id", "new_correct", "new_confidence", "correct_country"]].merge(
    img_group, on="image_id", how="left"
)
ga["any_correct"] = ga["any_correct"].fillna(0).astype(int)
ga["majority_correct"] = ga["majority_correct"].fillna(0).astype(int)
ga["avg_score"] = ga["avg_score"].fillna(0.0)
group_analysis = ga

group_accs = pd.Series({
    "EfficientNet-B0":  ga["new_correct"].mean(),
    "Group (any)":      ga["any_correct"].mean(),
    "Group (majority)": ga["majority_correct"].mean(),
    "Group (average)":  ga["avg_score"].mean(),
})

print("Accuracy comparison:")
for label, acc in group_accs.items():
    print(f"  {label:25s}: {acc:.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
colors_g = ["#1f77b4", "#2ca02c", "#ff7f00", "#d62728"]
(group_accs * 100).plot.bar(ax=ax, color=colors_g)
ax.set_ylabel("Accuracy (%)")
ax.set_title("Model vs. group aggregation methods (1,000 images)")
ax.set_ylim(0, max(group_accs) * 120)
for i, v in enumerate(group_accs * 100):
    ax.text(i, v + 0.8, f"{v:.1f}%", ha="center", fontsize=10, fontweight="bold")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# McNemar tests: group methods vs model
for label, hcol in [("Group (any correct)", "any_correct"),
                     ("Group (majority vote)", "majority_correct")]:
    b_v = int(((ga["new_correct"] == 1) & (ga[hcol] == 0)).sum())
    c_v = int(((ga["new_correct"] == 0) & (ga[hcol] == 1)).sum())
    test = mcnemar(b_v, c_v)
    print(f"{label} vs Model:")
    ct = pd.crosstab(
        ga[hcol].map({1: "group_right", 0: "group_wrong"}),
        ga["new_correct"].map({1: "model_right", 0: "model_wrong"}),
    )
    print(ct)
    print(f"  chi2={test['chi2']:.1f}, p={test['p']:.1e}")
    print()

## §C7 — Confusion Analysis

What countries does the model confuse most? The US/Canada pair is the model's
primary failure mode.

In [ ]:
wrong = images.loc[images["new_correct"] == 0].copy()
print(f"Total misclassifications: {len(wrong):,} of {len(images):,}")

top_confusions = (
    wrong.groupby(["correct_country", "new_prediction"])
    .size().rename("n").reset_index()
    .sort_values("n", ascending=False).head(15)
)
top_confusions

In [ ]:
top_countries = images["correct_country"].value_counts().head(15).index.tolist()
mask = images["correct_country"].isin(top_countries)
sub = images[mask]
ct_full = pd.crosstab(sub["correct_country"], sub["new_prediction"])
ct_top = ct_full.reindex(index=top_countries, columns=top_countries, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(ct_top, annot=True, fmt="d", cmap="Blues", ax=ax, cbar_kws={"shrink": 0.7})
ax.set_title("Confusion matrix: top 15 countries by frequency")
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

## §C8 — Visualising Statistical Significance

Comprehensive visualization of all McNemar test results.

In [ ]:
# Build a DataFrame of all McNemar results
sig_rows = []

for _, r in mcnemar_user.iterrows():
    sig_rows.append({
        "comparison": f"{r['user']} vs Model",
        "a_acc": r["human_acc"], "b_acc": r["model_acc"],
        "chi2": r["chi2"], "p_value": r["p_value"],
        "n": r["n"], "category": "Individual",
    })

for label, hcol in [("Group (any)", "any_correct"),
                     ("Group (majority)", "majority_correct")]:
    b_v = int(((ga["new_correct"] == 1) & (ga[hcol] == 0)).sum())
    c_v = int(((ga["new_correct"] == 0) & (ga[hcol] == 1)).sum())
    test = mcnemar(b_v, c_v)
    sig_rows.append({
        "comparison": f"{label} vs Model",
        "a_acc": ga[hcol].mean(), "b_acc": ga["new_correct"].mean(),
        "chi2": test["chi2"], "p_value": test["p"],
        "n": 1000, "category": "Group",
    })

b_m = int(((images["new_correct"] == 1) & (images["old_correct"] == 0)).sum())
c_m = int(((images["new_correct"] == 0) & (images["old_correct"] == 1)).sum())
test_old = mcnemar(b_m, c_m)
sig_rows.append({
    "comparison": "Old Model vs EfficientNet-B0",
    "a_acc": old_acc, "b_acc": new_acc,
    "chi2": test_old["chi2"], "p_value": test_old["p"],
    "n": 1000, "category": "Model",
})

sig_df = pd.DataFrame(sig_rows)
sig_df["neg_log10_p"] = -np.log10(sig_df["p_value"].clip(lower=1e-300))
sig_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors_sig = {"Individual": "#f58518", "Group": "#2ca02c", "Model": "#1f77b4"}
bar_colors = [colors_sig[c] for c in sig_df["category"]]

y_pos = np.arange(len(sig_df))
ax.barh(y_pos, sig_df["neg_log10_p"], color=bar_colors, edgecolor="white")
ax.set_yticks(y_pos)
ax.set_yticklabels(sig_df["comparison"])
ax.set_xlabel(r"$-\log_{10}(p)$  (higher = more significant)")
ax.set_title("McNemar test significance (all comparisons)")
ax.axvline(-np.log10(0.05), color="red", ls="--", lw=1, label="p = 0.05")
ax.axvline(-np.log10(0.001), color="red", ls=":", lw=1, label="p = 0.001")
for i, row_sig in sig_df.iterrows():
    ax.text(row_sig["neg_log10_p"] + 1, i, f"p = {row_sig['p_value']:.1e}", va="center", fontsize=8)
ax.legend(loc="lower right")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## §C9 — Rejecting the Null Hypothesis

This is the key result: the McNemar contingency table and discordant pair analysis
that formally tests our hypotheses.

- **H₀**: The model and the human group perform equally (symmetric discordant pairs)
- **Hₐ**: The model outperforms the human group

In [ ]:
both_right = int(((ga["any_correct"] == 1) & (ga["new_correct"] == 1)).sum())
model_only = int(((ga["any_correct"] == 0) & (ga["new_correct"] == 1)).sum())
human_only = int(((ga["any_correct"] == 1) & (ga["new_correct"] == 0)).sum())
both_wrong = int(((ga["any_correct"] == 0) & (ga["new_correct"] == 0)).sum())
test_final = mcnemar(model_only, human_only)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: 2x2 contingency table heatmap
table = np.array([[both_right, human_only],
                  [model_only, both_wrong]])
labels_ht = np.array([[f"Both correct\n{both_right}", f"Only humans correct\n{human_only}"],
                      [f"Only model correct\n{model_only}", f"Both wrong\n{both_wrong}"]])
sns.heatmap(table, annot=labels_ht, fmt="", cmap="Blues", cbar=False,
            xticklabels=["Model correct", "Model wrong"],
            yticklabels=["Humans correct", "Humans wrong"],
            ax=axes[0], annot_kws={"fontsize": 12})
axes[0].set_title("McNemar contingency table\n(1,000 images, group = any participant correct)", fontsize=11)

# Right: discordant pair imbalance
disc_labels = ["Model correct,\nhumans wrong", "Humans correct,\nmodel wrong"]
disc_vals = [model_only, human_only]
bars_disc = axes[1].bar(disc_labels, disc_vals, color=["#1f77b4", "#f58518"], width=0.5)
axes[1].bar_label(bars_disc, fontsize=13, fontweight="bold", padding=4)

axes[1].set_title(
    f"Discordant pairs (drives the McNemar test)\n"
    f"$\\chi^2$ = {test_final['chi2']:.1f},  p = {test_final['p']:.1e}",
    fontsize=11
)
axes[1].set_ylabel("number of images")

if test_final["p"] < 0.05:
    axes[1].text(0.5, 0.85,
                 "Reject $H_0$ at $\\alpha$ = 0.05",
                 transform=axes[1].transAxes, ha="center",
                 fontsize=13, fontweight="bold", color="darkred",
                 bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", edgecolor="darkred"))

plt.tight_layout()
plt.show()

print(f"The model got {model_only} images right that the group missed.")
print(f"The group got {human_only} images right that the model missed.")
print(f"McNemar chi-squared = {test_final['chi2']:.1f}, p = {test_final['p']:.1e}")
print()
print("We REJECT the null hypothesis at alpha = 0.05.")
print("The EfficientNet-B0 model significantly outperforms the human case study group.")

## §C10 — Summary Table

In [ ]:
summary_rows = [
    {"label": "Old model", "accuracy": old_acc, "n": len(images)},
    {"label": "EfficientNet-B0", "accuracy": new_acc, "n": len(images)},
]
for u in user_summary.index:
    summary_rows.append({
        "label": u,
        "accuracy": user_summary.loc[u, "accuracy"],
        "n": int(user_summary.loc[u, "guesses"]),
    })
summary_rows.extend([
    {"label": "Group (any correct)", "accuracy": group_accs["Group (any)"], "n": 1000},
    {"label": "Group (majority vote)", "accuracy": group_accs["Group (majority)"], "n": 1000},
    {"label": "Group (average score)", "accuracy": group_accs["Group (average)"], "n": 1000},
    {"label": "Random baseline (1/161)", "accuracy": 1 / 161, "n": None},
])

summary = pd.DataFrame(summary_rows).set_index("label")
summary["accuracy_pct"] = (summary["accuracy"] * 100).round(2)
summary

---
## Conclusion

Our EfficientNet-B0 model, fine-tuned on ~500,000 street-view images from OpenStreetView-5M,
achieved **64.7% accuracy** on the 1,000 test images at identifying the country of origin —
significantly outperforming the human case study group under every aggregation method:

- **Individual participants**: 8.7% – 15.3% accuracy (all p < 10⁻⁴⁰)
- **Group (any correct)**: 20.6% accuracy (p < 10⁻⁷⁰)
- **Group (majority vote)**: 12.7% accuracy

All McNemar tests reject H₀ at α = 0.05 with overwhelming significance.

The model also improved by +32.6 pp from the old checkpoint (32.1%) to the final
EfficientNet-B0 (64.7%), achieved through training on more data (10 shards / ~500K images
vs. 5 shards / ~30K images).

> **Note on accuracy figures:** The 64.7% reported here is the model's accuracy on the
> 1,000 webapp test images used for the human comparison. On the full held-out evaluation
> set from training, the model achieved 57.6% top-1 accuracy (see Part A). The difference
> reflects sample variation between the two test sets.

**Key limitations**: The model shows country-level bias toward well-represented countries
(US, Russia) and struggles with geographically adjacent pairs (US/Canada confusion).

**Future work**: Train on the full 5M-image dataset, implement distance-based scoring
(haversine) rather than binary country matching, and explore larger architectures
(EfficientNet-B4, ConvNeXt).